In [ ]:
#cell 1
# Install runtime dependencies for Gemma 4 + vLLM on Colab / Blackwell.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy pandas accelerate safetensors huggingface_hub

# Remove optional packages that may break imports in some Colab/vLLM environments.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Gemma 4 needs recent Transformers support.
!uv pip install --system -U "transformers>=5.5.0"

# Recent vLLM nightly for CUDA 13 / Blackwell.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 71 packages in 99ms
Prepared 8 packages in 0.32ms
Uninstalled 8 packages in 126ms
Installed 8 packages in 117ms
 - numpy==2.3.5
 + numpy==2.5.0
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.1
 - triton==3.6.0
 + triton==3.7.1
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 48ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 70ms
Checked 27 packages in 0.25ms
Using Python 3.12.13 environment at: /usr
Resolved 190 packages in 6.64s
Prepared 10 packages in 16ms
Uninstalled 8 packages in 112ms
Installed 10 packages in 107ms
 - numpy==2.5.0
 + numpy==2.3.5
 - nvidia-cublas==

In [ ]:
#cell 2
# Imports and global configuration for PyRAG answer generation with Gemma 4.

import os
import re
import gc
import json
import time
import shlex
import shutil
import psutil
import subprocess
import traceback

from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI
from transformers import AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Gemma 4 model from the RAG Gemma notebook.
LLM_MODEL_NAME = "google/gemma-4-26B-A4B-it"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# The PyRAG answer prompt sent to the model is capped around 8192 tokens,
# matching your other answer notebooks.
MAX_INPUT_TOKENS = 8192

# Non-thinking answer budget.
# PyRAG asks for a short <redacted_thinking> block plus a short <answer>.
ANSWER_MAX_TOKENS = 256

# Gemma 4 26B-A4B is launched with long context in the RAG Gemma notebook.
# The actual answer prompt is still capped by MAX_INPUT_TOKENS for fairness.
MAX_MODEL_LEN = 131072

# vLLM recommends roughly 0.90-0.95. Keep some safety margin for Colab.
GPU_MEMORY_UTILIZATION = 0.92

MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = 32768

SERVER_LOG_PATH = Path("/content/vllm_gemma4_pyrag_answer_server.log")
SERVER_PID_PATH = Path("/content/vllm_gemma4_pyrag_answer_server.pid")

PROJECT_DIR = Path("/content/drive/MyDrive/final_project")
PYRAG_DIR = PROJECT_DIR / "pyrag"

# Your PyRAG evidence files are saved directly under /pyrag.
DRIVE_EVIDENCE_DIR = PYRAG_DIR
DRIVE_ANSWER_DIR = PYRAG_DIR / "answers"

LOCAL_RUNTIME_DIR = Path("/content/final_project_pyrag_gemma4_answer_copy")
LOCAL_EVIDENCE_DIR = LOCAL_RUNTIME_DIR / "evidence"
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/ali-mohmmadi/PyRAG.git"
REPO_DIR = Path("/content/PyRAG")

# Fair-comparison cap: the answer step must never see more than 6 chunks.
MAX_EVIDENCE_CHUNKS = 6

# True = closer to the PyRAG baseline style:
#   1) answer with retrieved docs using PyRAG's WITH_DOCS prompt
#   2) synthesize the final answer from the produced facts using PyRAG's NO_DOCS prompt
# False = one PyRAG WITH_DOCS answer call only.
USE_REPOSITORY_TWO_STAGE = True

DATASETS = {
    "hotpotqa": {
        "drive_evidence_path": DRIVE_EVIDENCE_DIR / "hotpotqa_evidence.json",
        "local_evidence_path": LOCAL_EVIDENCE_DIR / "hotpotqa_evidence.json",
        "answer_path": DRIVE_ANSWER_DIR / "hotpotqa_gemma4_answers.json",
    },
    "2wikimultihopqa": {
        "drive_evidence_path": DRIVE_EVIDENCE_DIR / "2wikimultihopqa_evidence.json",
        "local_evidence_path": LOCAL_EVIDENCE_DIR / "2wikimultihopqa_evidence.json",
        "answer_path": DRIVE_ANSWER_DIR / "2wikimultihopqa_gemma4_answers.json",
    },
}

EXPECTED_NUM_RECORDS_PER_DATASET = 1000

# Use None to process all records.
ANSWER_START_INDEX = 0
ANSWER_END_INDEX = None

SAVE_EVERY_N = 1
CLEAR_CACHE_EVERY_N = 25

# Resume from existing Gemma 4 PyRAG answer files if they already exist.
RESUME_IF_EXISTS = True

print("Model:", LLM_MODEL_NAME)
print("Non-thinking mode:", True)
print("Max input tokens:", MAX_INPUT_TOKENS)
print("Answer max tokens:", ANSWER_MAX_TOKENS)
print("Max model len:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Max num batched tokens:", MAX_NUM_BATCHED_TOKENS)
print("PyRAG directory:", PYRAG_DIR)
print("Evidence directory:", DRIVE_EVIDENCE_DIR)
print("Answer directory:", DRIVE_ANSWER_DIR)
print("Repository URL:", REPO_URL)
print("Repository two-stage answer:", USE_REPOSITORY_TWO_STAGE)
print("Max evidence chunks visible to answer model:", MAX_EVIDENCE_CHUNKS)

for dataset_name, cfg in DATASETS.items():
    print("=" * 80)
    print("Dataset:", dataset_name)
    print("Drive evidence:", cfg["drive_evidence_path"])
    print("Local evidence:", cfg["local_evidence_path"])
    print("Answer output:", cfg["answer_path"])

Model: google/gemma-4-26B-A4B-it
Non-thinking mode: True
Max input tokens: 8192
Answer max tokens: 256
Max model len: 131072
GPU memory utilization: 0.92
Max num batched tokens: 32768
PyRAG directory: /content/drive/MyDrive/final_project/pyrag
Evidence directory: /content/drive/MyDrive/final_project/pyrag
Answer directory: /content/drive/MyDrive/final_project/pyrag/answers
Repository URL: https://github.com/ali-mohmmadi/PyRAG.git
Repository two-stage answer: True
Max evidence chunks visible to answer model: 6
Dataset: hotpotqa
Drive evidence: /content/drive/MyDrive/final_project/pyrag/hotpotqa_evidence.json
Local evidence: /content/final_project_pyrag_gemma4_answer_copy/evidence/hotpotqa_evidence.json
Answer output: /content/drive/MyDrive/final_project/pyrag/answers/hotpotqa_gemma4_answers.json
Dataset: 2wikimultihopqa
Drive evidence: /content/drive/MyDrive/final_project/pyrag/2wikimultihopqa_evidence.json
Local evidence: /content/final_project_pyrag_gemma4_answer_copy/evidence/2wikimu

In [ ]:
#cell 3
# Mount Google Drive and copy PyRAG evidence files to local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
assert PYRAG_DIR.exists(), f"PYRAG_DIR does not exist: {PYRAG_DIR}"
assert DRIVE_EVIDENCE_DIR.exists(), f"Evidence directory does not exist: {DRIVE_EVIDENCE_DIR}"

DRIVE_ANSWER_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)

def file_is_same_size(src: Path, dst: Path) -> bool:
    """Check whether the destination file exists and has the same size as the source."""
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    """Copy a file to local disk using a temporary file to avoid partial copies."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print(f"Local copy already exists: {dst}")
        return

    tmp = dst.with_name(dst.name + ".tmp")

    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

    print(f"Copied to local disk: {src} -> {dst}")

for dataset_name, cfg in DATASETS.items():
    drive_path = cfg["drive_evidence_path"]
    local_path = cfg["local_evidence_path"]

    if not drive_path.exists():
        raise FileNotFoundError(f"{dataset_name}: PyRAG evidence file not found: {drive_path}")

    copy_file_to_local(drive_path, local_path)

    print(f"{dataset_name}: local evidence size MB:", local_path.stat().st_size / (1024 ** 2))

print("Answer directory is ready:", DRIVE_ANSWER_DIR)

Mounted at /content/drive
Copied to local disk: /content/drive/MyDrive/final_project/pyrag/hotpotqa_evidence.json -> /content/final_project_pyrag_gemma4_answer_copy/evidence/hotpotqa_evidence.json
hotpotqa: local evidence size MB: 9.70973014831543
Copied to local disk: /content/drive/MyDrive/final_project/pyrag/2wikimultihopqa_evidence.json -> /content/final_project_pyrag_gemma4_answer_copy/evidence/2wikimultihopqa_evidence.json
2wikimultihopqa: local evidence size MB: 7.817546844482422
Answer directory is ready: /content/drive/MyDrive/final_project/pyrag/answers


In [ ]:
#cell 4
# Load and validate PyRAG evidence files.

def load_json_list(json_path: Path, dataset_name: str) -> List[Dict[str, Any]]:
    """Load a JSON file whose root must be a list."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"{dataset_name}: JSON root must be a list.")

    return data

def validate_evidence_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one PyRAG evidence record."""
    required_keys = {"type", "question", "answer", "evidence_chunk"}

    if not isinstance(record, dict):
        raise ValueError(f"{dataset_name}: record {index} is not a dictionary.")

    missing = required_keys - set(record.keys())
    if missing:
        raise ValueError(f"{dataset_name}: record {index} is missing keys: {missing}")

    if not isinstance(record["question"], str) or not record["question"].strip():
        raise ValueError(f"{dataset_name}: record {index} has an empty question.")

    if not isinstance(record["evidence_chunk"], list):
        raise ValueError(f"{dataset_name}: record {index} evidence_chunk must be a list.")

    if len(record["evidence_chunk"]) == 0:
        raise ValueError(f"{dataset_name}: record {index} has no evidence chunks.")

    if len(record["evidence_chunk"]) > MAX_EVIDENCE_CHUNKS:
        raise ValueError(
            f"{dataset_name}: record {index} has {len(record['evidence_chunk'])} chunks; "
            f"the fair-comparison cap is {MAX_EVIDENCE_CHUNKS}."
        )

    for j, chunk in enumerate(record["evidence_chunk"]):
        if not isinstance(chunk, dict):
            raise ValueError(f"{dataset_name}: record {index}, chunk {j} is not a dictionary.")

        if "title" not in chunk or "text" not in chunk:
            raise ValueError(
                f"{dataset_name}: record {index}, chunk {j} must contain title and text."
            )

def load_and_validate_dataset(dataset_name: str, evidence_path: Path) -> List[Dict[str, Any]]:
    """Load and validate one dataset evidence file."""
    records = load_json_list(evidence_path, dataset_name)

    if len(records) != EXPECTED_NUM_RECORDS_PER_DATASET:
        print(
            f"Warning: {dataset_name} has {len(records)} records, "
            f"expected {EXPECTED_NUM_RECORDS_PER_DATASET}."
        )

    chunk_counts = []

    for i, rec in enumerate(records):
        validate_evidence_record(rec, dataset_name, i)
        chunk_counts.append(len(rec["evidence_chunk"]))

    print("=" * 80)
    print(f"{dataset_name}: validation passed.")
    print("Number of records:", len(records))
    print("Chunk count min/max:", min(chunk_counts), max(chunk_counts))
    print("First question:", records[0]["question"])
    print("First GT answer:", records[0]["answer"])
    print("First number of chunks:", len(records[0]["evidence_chunk"]))
    print("First chunk keys:", list(records[0]["evidence_chunk"][0].keys()))

    return records

evidence_data = {}

for dataset_name, cfg in DATASETS.items():
    evidence_data[dataset_name] = load_and_validate_dataset(
        dataset_name=dataset_name,
        evidence_path=cfg["local_evidence_path"],
    )

hotpotqa: validation passed.
Number of records: 1000
Chunk count min/max: 6 6
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First GT answer: Bedknobs and Broomsticks
First number of chunks: 6
First chunk keys: ['title', 'text']
2wikimultihopqa: validation passed.
Number of records: 1000
Chunk count min/max: 6 6
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First GT answer: Kamakalawa
First number of chunks: 6
First chunk keys: ['title', 'text']


In [ ]:
#cell 4.5
# Clone/update PyRAG and import the repository's answer prompt utilities.
#
# Insert this cell between cell 4 and cell 5.

import sys
import subprocess

if not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=False)
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

os.environ["PYTHONPATH"] = str(REPO_DIR) + ":" + os.environ.get("PYTHONPATH", "")

from pyrag.tools import (
    ANSWER_SYSTEM_PROMPT_WITH_DOCS,
    ANSWER_SYSTEM_PROMPT_NO_DOCS,
)
from pyrag.utils import (
    extract_answer_tag,
    format_docs_for_prompt,
)

print("Repository ready:", REPO_DIR)
subprocess.run(["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"], check=False)

print("Imported PyRAG answer prompts and utilities.")
print("WITH_DOCS prompt preview:", ANSWER_SYSTEM_PROMPT_WITH_DOCS[:250].replace("\n", " "))
print("NO_DOCS prompt preview:", ANSWER_SYSTEM_PROMPT_NO_DOCS[:250].replace("\n", " "))

Repository ready: /content/PyRAG
Imported PyRAG answer prompts and utilities.
WITH_DOCS prompt preview: You are given a question and retrieved documents. You MUST answer using ONLY information from the retrieved documents. Even for yes/no questions, decide yes or no by reasoning from facts in the documents.  Output format (STRICT): <redacted_thinking> 
NO_DOCS prompt preview: There are NO retrieved documents. The question text itself contains background facts (after 'Given:') and the actual question to answer (after 'Answer the question:'). You MUST use the provided facts to answer the ACTUAL QUESTION.  CRITICAL: Your job


In [ ]:
#cell 5
# Start Gemma 4 vLLM server in non-thinking mode.

def kill_process_tree(pid: int) -> None:
    """Kill a process and all child processes."""
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

def find_or_download_gemma4_chat_template() -> Optional[Path]:
    """Find or download the Gemma 4 vLLM chat template."""
    template_name = "tool_chat_template_gemma4.jinja"
    local_template = Path("/content") / template_name

    if local_template.exists() and local_template.stat().st_size > 0:
        return local_template

    search_roots = []

    try:
        import vllm as vllm_pkg
        vllm_path = Path(vllm_pkg.__file__).resolve()
        search_roots.extend([
            vllm_path.parent,
            vllm_path.parent.parent,
        ])
    except Exception:
        pass

    search_roots.extend([
        Path("/usr/local/lib/python3.12/dist-packages"),
        Path("/usr/local/lib/python3.12/site-packages"),
        Path("/usr/lib/python3.12/site-packages"),
        Path("/content"),
    ])

    for root in search_roots:
        if not root.exists():
            continue

        try:
            matches = sorted(root.rglob(template_name))
        except Exception:
            matches = []

        for match in matches:
            if match.exists() and match.stat().st_size > 0:
                print("Found Gemma 4 chat template:", match)
                return match

    # Fallback for pip installs where examples are not packaged.
    try:
        import requests

        url = (
            "https://raw.githubusercontent.com/vllm-project/vllm/main/"
            "examples/tool_chat_template_gemma4.jinja"
        )
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        local_template.write_text(r.text, encoding="utf-8")

        if local_template.exists() and local_template.stat().st_size > 0:
            print("Downloaded Gemma 4 chat template:", local_template)
            return local_template

    except Exception as e:
        print("Warning: could not download Gemma 4 chat template:", repr(e))

    return None

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(int(old_pid))

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

gemma4_chat_template_path = find_or_download_gemma4_chat_template()

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only QA workload.
    "--language-model-only",
    "--limit-mm-per-prompt", '{"image": 0, "audio": 0}',

    # Gemma 4 protocol support.
    # The model is launched with enable_thinking=false for non-thinking mode.
    "--reasoning-parser", "gemma4",
    "--tool-call-parser", "gemma4",
    "--enable-auto-tool-choice",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--enable-prefix-caching",
    "--generation-config", "vllm",
    "--dtype", "bfloat16",

    # Force Triton MoE backend to avoid FlashInfer CUTLASS MoE JIT crash on some vLLM nightlies.
    "--moe-backend", "triton",

    "--trust-remote-code",
]

if gemma4_chat_template_path is not None:
    cmd.extend(["--chat-template", str(gemma4_chat_template_path)])
else:
    print("Warning: running without explicit Gemma 4 tool chat template.")

server_env = os.environ.copy()

# Blackwell / CUDA 13 fixes.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Downloaded Gemma 4 chat template: /content/tool_chat_template_gemma4.jinja
Command:
vllm serve google/gemma-4-26B-A4B-it --host 0.0.0.0 --port 8000 --max-model-len 131072 --gpu-memory-utilization 0.92 --language-model-only --limit-mm-per-prompt '{"image": 0, "audio": 0}' --reasoning-parser gemma4 --tool-call-parser gemma4 --enable-auto-tool-choice --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 1 --max-num-batched-tokens 32768 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --moe-backend triton --trust-remote-code --chat-template /content/tool_chat_template_gemma4.jinja

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 2587
Log: /content/vllm_gemma4_pyrag_answer_server.log


In [ ]:
#cell 6
# Wait for vLLM server and create an OpenAI-compatible client.

import requests

def tail_log(path: Path, n: int = 80) -> str:
    """Read the last n log lines."""
    if not path.exists():
        return ""

    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False
SERVER_MODEL_ID = None

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()
    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                SERVER_MODEL_ID = model_info["id"]
                print("vLLM server is ready.")
                print("Model:", SERVER_MODEL_ID)
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)

        if recent.strip():
            print(recent)

        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# Deterministic decoding is better for QA evaluation.
LLM_SAMPLING_KWARGS = {
    "temperature": 0.0,
    "top_p": 1.0,
    "presence_penalty": 0.0,
}

# Non-thinking mode is also sent per request.
LLM_EXTRA_BODY = {
    "chat_template_kwargs": {
        "enable_thinking": False,
    },
}

print("OpenAI-compatible client is ready.")
print("Server model id:", SERVER_MODEL_ID)
print("Non-thinking extra body:", LLM_EXTRA_BODY)
print("Sampling kwargs:", LLM_SAMPLING_KWARGS)

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=3065) INFO 06-28 20:21:26 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=3065) INFO 06-28 20:21:26 [parallel_state.py:1588] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:59383 backend=nccl
(EngineCore pid=3065) INFO 06-28 20:21:26 [parallel_state.py:1923] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank 0, EPLB rank N/A
(EngineCore pid=3065) INFO 06-28 20:21:26 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=3065) INFO 06-28 20:21:26 [gpu_model_runner.py:5168] Starting to load model google/gemma-4-26B-A4B-it...
(EngineCore pid=3065) INFO 06-28 20:21:27 [vllm.py:1006] Asynchronous scheduling is enabled.
(EngineCore pid=3065) INFO 06-28 20:

In [ ]:
#cell 7
# Load tokenizer and define PyRAG-style prompt/context builders.

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

TAG_RE_TEMPLATE = r"<{tag}>\s*(.*?)\s*</{tag}>"

def count_tokens(text: str) -> int:
    """Count tokens without adding special tokens."""
    return len(tokenizer.encode(text, add_special_tokens=False))

def count_message_tokens(messages: List[Dict[str, str]]) -> int:
    """Approximate chat input tokens by counting message contents."""
    return sum(count_tokens(m.get("content", "")) for m in messages)

def extract_xml_tag(text: str, tag: str) -> str:
    """Extract a simple XML-like tag span from model output."""
    if text is None:
        return ""

    pattern = TAG_RE_TEMPLATE.format(tag=re.escape(tag))
    match = re.search(pattern, str(text), flags=re.DOTALL | re.IGNORECASE)
    if not match:
        return ""

    return match.group(1).strip()

def strip_model_artifacts(text: str) -> str:
    """Remove accidental thinking blocks, Gemma turn markers, and formatting artifacts."""
    if text is None:
        return ""

    text = str(text)

    # Remove possible XML-style thinking blocks.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<think>", "").replace("</think>", "")

    # Remove common model special tokens while preserving PyRAG tags such as <answer>.
    for token in [
        "<start_of_turn>",
        "<end_of_turn>",
        "<bos>",
        "<eos>",
        "<pad>",
    ]:
        text = text.replace(token, "")

    text = re.sub(r"<\|[^>]+?\|>", "", text)

    # Remove code fences if the model accidentally uses them.
    text = text.replace("```json", "").replace("```text", "").replace("```", "")

    return text.strip()

def clean_extracted_answer(text: str) -> str:
    """Clean the answer extracted from a PyRAG <answer> block."""
    text = strip_model_artifacts(text)
    text = extract_answer_tag(text)
    text = strip_model_artifacts(text)

    text = re.sub(r"^\s*ANSWER\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"^\s*FINAL\s*:\s*", "", text, flags=re.IGNORECASE).strip()

    # Keep only the first non-empty line if the model falls back to a verbose answer.
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0]

    return text.strip().strip('"').strip("'").strip()

def evidence_chunks_to_pyrag_docs(evidence_chunks: List[Dict[str, Any]]) -> List[str]:
    """
    Convert evidence_chunk objects to the same document-string shape used by PyRAG:
    Doc i (Title: ...)
    body
    """
    docs = []

    for idx, chunk in enumerate(evidence_chunks[:MAX_EVIDENCE_CHUNKS], start=1):
        title = str(chunk.get("title", "")).strip().strip('"')
        text = str(chunk.get("text", "")).strip()

        docs.append(
            f"Doc {idx} (Title: {title})\n{text}"
        )

    return docs

def build_pyrag_user_prompt(question: str, docs_text: str) -> str:
    """Build the user prompt used by PyRAG's answer() tool."""
    return (
        "=== QUESTION ===\n"
        f"{str(question).strip()}\n"
        "=== END QUESTION ===\n\n"
        "=== RETRIEVED DOCUMENTS ===\n"
        f"{docs_text}\n"
        "=== END DOCUMENTS ==="
    )

def build_docs_prompt(question: str, evidence_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Build the first-stage PyRAG WITH_DOCS prompt.
    This uses only question + evidence_chunk and never uses gold answer/supports.
    """
    docs = evidence_chunks_to_pyrag_docs(evidence_chunks)
    docs_text = format_docs_for_prompt(docs)

    system_prompt = ANSWER_SYSTEM_PROMPT_WITH_DOCS
    user_prompt = build_pyrag_user_prompt(question, docs_text)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    original_tokens = count_message_tokens(messages)

    if original_tokens <= MAX_INPUT_TOKENS:
        return {
            "messages": messages,
            "docs": docs,
            "input_tokens": original_tokens,
            "was_truncated": False,
        }

    # Truncate only the retrieved-document block. Keep the question and PyRAG system prompt intact.
    empty_user_prompt = build_pyrag_user_prompt(question, "")
    overhead_tokens = count_message_tokens([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": empty_user_prompt},
    ])

    docs_budget = max(0, MAX_INPUT_TOKENS - overhead_tokens - 16)

    docs_ids = tokenizer.encode(docs_text, add_special_tokens=False)
    truncated_docs_ids = docs_ids[:docs_budget]
    truncated_docs_text = tokenizer.decode(truncated_docs_ids, skip_special_tokens=True)

    truncated_messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": build_pyrag_user_prompt(question, truncated_docs_text)},
    ]

    return {
        "messages": truncated_messages,
        "docs": docs,
        "input_tokens": count_message_tokens(truncated_messages),
        "was_truncated": True,
        "original_input_tokens": original_tokens,
    }

def build_facts_from_docs_response(raw_docs_response: str) -> str:
    """
    Convert the first-stage PyRAG output into a compact facts block for the
    second-stage NO_DOCS synthesis prompt.
    """
    raw_docs_response = strip_model_artifacts(raw_docs_response)

    redacted_thinking = extract_xml_tag(raw_docs_response, "redacted_thinking")
    intermediate_answer = clean_extracted_answer(raw_docs_response)

    parts = []

    if redacted_thinking:
        parts.append(redacted_thinking)

    if intermediate_answer:
        parts.append(f"Intermediate answer: {intermediate_answer}")

    if not parts:
        # Fallback: use the raw output if the model did not follow the expected tags.
        fallback = raw_docs_response.strip()
        if fallback:
            parts.append(fallback)

    return "\n".join(parts).strip()

def build_synthesis_prompt(question: str, facts: str) -> Dict[str, Any]:
    """
    Build the second-stage PyRAG NO_DOCS prompt:
    Given: facts
    Answer the question: original question
    """
    synth_query = (
        "Given:\n"
        f"{str(facts).strip()}\n\n"
        "Answer the question:\n"
        f"{str(question).strip()}"
    ).strip()

    system_prompt = ANSWER_SYSTEM_PROMPT_NO_DOCS
    user_prompt = build_pyrag_user_prompt(
        question=synth_query,
        docs_text=format_docs_for_prompt([]),
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    original_tokens = count_message_tokens(messages)

    if original_tokens <= MAX_INPUT_TOKENS:
        return {
            "messages": messages,
            "synthesis_query": synth_query,
            "input_tokens": original_tokens,
            "was_truncated": False,
        }

    # Truncate only the facts block if needed.
    shell_query = (
        "Given:\n\n"
        "Answer the question:\n"
        f"{str(question).strip()}"
    ).strip()

    shell_user_prompt = build_pyrag_user_prompt(
        question=shell_query,
        docs_text=format_docs_for_prompt([]),
    )

    overhead_tokens = count_message_tokens([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": shell_user_prompt},
    ])

    facts_budget = max(0, MAX_INPUT_TOKENS - overhead_tokens - 16)

    facts_ids = tokenizer.encode(str(facts).strip(), add_special_tokens=False)
    truncated_facts = tokenizer.decode(facts_ids[:facts_budget], skip_special_tokens=True)

    truncated_synth_query = (
        "Given:\n"
        f"{truncated_facts.strip()}\n\n"
        "Answer the question:\n"
        f"{str(question).strip()}"
    ).strip()

    truncated_messages = [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": build_pyrag_user_prompt(
                question=truncated_synth_query,
                docs_text=format_docs_for_prompt([]),
            ),
        },
    ]

    return {
        "messages": truncated_messages,
        "synthesis_query": truncated_synth_query,
        "input_tokens": count_message_tokens(truncated_messages),
        "was_truncated": True,
        "original_input_tokens": original_tokens,
    }

# Sanity check: make sure the prompt uses only question and evidence_chunk.
sample_dataset = "hotpotqa"
sample_record = evidence_data[sample_dataset][0]
sample_prompt_info = build_docs_prompt(
    question=sample_record["question"],
    evidence_chunks=sample_record["evidence_chunk"],
)

print("Sample dataset:", sample_dataset)
print("Sample input tokens:", sample_prompt_info["input_tokens"])
print("Sample was truncated:", sample_prompt_info["was_truncated"])
print("Sample number of docs visible to LLM:", len(sample_prompt_info["docs"]))

print("\nSample system prompt preview:")
print(sample_prompt_info["messages"][0]["content"][:1200])

print("\nSample user prompt preview:")
print(sample_prompt_info["messages"][1]["content"][:2000])

Sample dataset: hotpotqa
Sample input tokens: 2756
Sample was truncated: False
Sample number of docs visible to LLM: 6

Sample system prompt preview:
You are given a question and retrieved documents.
You MUST answer using ONLY information from the retrieved documents.
Even for yes/no questions, decide yes or no by reasoning from facts in the documents.

Output format (STRICT):
<redacted_thinking> ... </redacted_thinking>
<answer> ... </answer>

Evidence citation rule:
- Whenever you use evidence from the documents in your reasoning, you MUST cite it inline as Doc [i] (matching the document indices shown in the retrieved block, e.g. [Doc 1] → Doc [1]).
- Only cite documents that are actually relevant.
- Keep <redacted_thinking> concise (1–3 sentences).

Answer rules:
- The <answer> should be a short phrase, preferably taken directly from the documents when possible.
- Match the answer TYPE to the QUESTION: WHO / which person / 谁先 / born first / earlier → a person's NAME in <answer>, not

In [ ]:
#cell 8
# Define Gemma 4 LLM calls, PyRAG response cleanup, and JSON saving helpers.

def extract_chat_message_content(completion: Any) -> str:
    """Extract final message content from an OpenAI-compatible chat completion."""
    message = completion.choices[0].message

    content = getattr(message, "content", None)

    if content is None and isinstance(message, dict):
        content = message.get("content")

    # Only the final assistant content is used as the response.
    if content is None:
        content = ""

    return str(content)

def call_chat_completion(messages: List[Dict[str, str]], max_retries: int = 3) -> str:
    """Call the local Gemma 4 vLLM server and return raw final model text."""
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            completion = client.chat.completions.create(
                model=SERVER_MODEL_ID or LLM_MODEL_NAME,
                messages=messages,
                max_tokens=ANSWER_MAX_TOKENS,
                **LLM_SAMPLING_KWARGS,
                extra_body=LLM_EXTRA_BODY,
            )

            raw_text = extract_chat_message_content(completion)

            if raw_text is not None and str(raw_text).strip():
                return str(raw_text)

            print(f"Warning: empty final content on attempt {attempt}/{max_retries}. Retrying...")
            time.sleep(2 * attempt)

        except Exception as e:
            last_error = e
            print(f"LLM call failed on attempt {attempt}/{max_retries}: {repr(e)}")
            time.sleep(2 * attempt)

    if last_error is not None:
        raise RuntimeError(f"LLM call failed after {max_retries} attempts: {repr(last_error)}")

    return ""

def answer_with_pyrag_docs(question: str, evidence_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """First PyRAG-style stage: answer from retrieved documents."""
    prompt_info = build_docs_prompt(
        question=question,
        evidence_chunks=evidence_chunks,
    )

    raw = call_chat_completion(prompt_info["messages"])
    answer = clean_extracted_answer(raw)
    facts = build_facts_from_docs_response(raw)

    return {
        "answer": answer,
        "raw": raw,
        "facts": facts,
        "input_tokens": prompt_info["input_tokens"],
        "was_truncated": prompt_info.get("was_truncated", False),
        "original_input_tokens": prompt_info.get("original_input_tokens"),
        "num_docs": len(prompt_info["docs"]),
    }

def synthesize_with_pyrag_facts(question: str, facts: str) -> Dict[str, Any]:
    """Second PyRAG-style stage: synthesize final answer from facts."""
    prompt_info = build_synthesis_prompt(
        question=question,
        facts=facts,
    )

    raw = call_chat_completion(prompt_info["messages"])
    answer = clean_extracted_answer(raw)

    return {
        "answer": answer,
        "raw": raw,
        "synthesis_query": prompt_info["synthesis_query"],
        "input_tokens": prompt_info["input_tokens"],
        "was_truncated": prompt_info.get("was_truncated", False),
        "original_input_tokens": prompt_info.get("original_input_tokens"),
    }

def call_llm_answer(question: str, evidence_chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Return the final answer using the chosen PyRAG-style answer path.

    Default:
    - WITH_DOCS prompt over <=6 evidence chunks
    - NO_DOCS synthesis prompt over the generated facts
    """
    docs_stage = answer_with_pyrag_docs(
        question=question,
        evidence_chunks=evidence_chunks,
    )

    if USE_REPOSITORY_TWO_STAGE:
        synthesis_stage = synthesize_with_pyrag_facts(
            question=question,
            facts=docs_stage["facts"],
        )

        return {
            "response": synthesis_stage["answer"],
            "docs_stage_answer": docs_stage["answer"],
            "docs_stage_facts": docs_stage["facts"],
            "docs_stage_raw": docs_stage["raw"],
            "synthesis_raw": synthesis_stage["raw"],
            "total_input_tokens": docs_stage["input_tokens"] + synthesis_stage["input_tokens"],
            "was_truncated": docs_stage["was_truncated"] or synthesis_stage["was_truncated"],
            "num_docs": docs_stage["num_docs"],
        }

    return {
        "response": docs_stage["answer"],
        "docs_stage_answer": docs_stage["answer"],
        "docs_stage_facts": docs_stage["facts"],
        "docs_stage_raw": docs_stage["raw"],
        "synthesis_raw": None,
        "total_input_tokens": docs_stage["input_tokens"],
        "was_truncated": docs_stage["was_truncated"],
        "num_docs": docs_stage["num_docs"],
    }

def atomic_write_json(data: Any, path: Path) -> None:
    """Atomically save JSON to the target path."""
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_name(path.name + ".tmp")

    with open(tmp_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    os.replace(tmp_path, path)

def load_existing_answers(answer_path: Path) -> List[Dict[str, Any]]:
    """Load existing answers for resume mode."""
    if not answer_path.exists():
        return []

    with open(answer_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError(f"Existing answer file must contain a list: {answer_path}")

    return data

def validate_answer_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one answer output record."""
    required_keys = {"type", "question", "gt", "response"}

    if set(record.keys()) != required_keys:
        raise ValueError(
            f"{dataset_name}: answer record {index} must have exactly {required_keys}, "
            f"but found {set(record.keys())}"
        )

print("PyRAG-style Gemma 4 helper functions are ready.")
print("Saved answer JSON keys will stay exactly: type, question, gt, response.")

PyRAG-style Gemma 4 helper functions are ready.
Saved answer JSON keys will stay exactly: type, question, gt, response.


In [ ]:
#cell 9
# Test one PyRAG-style Gemma 4 LLM answer before running the full datasets.

test_dataset = "hotpotqa"
test_record = evidence_data[test_dataset][0]

test_prompt_info = build_docs_prompt(
    question=test_record["question"],
    evidence_chunks=test_record["evidence_chunk"],
)

print("Test dataset:", test_dataset)
print("Test question:", test_record["question"])
print("Test GT answer:", test_record["answer"])
print("Note: GT is printed only for human checking and is not sent to the LLM.")
print("Test docs-stage input tokens:", test_prompt_info["input_tokens"])
print("Test docs-stage was truncated:", test_prompt_info["was_truncated"])
print("Test number of evidence chunks:", len(test_record["evidence_chunk"]))

test_result = call_llm_answer(
    question=test_record["question"],
    evidence_chunks=test_record["evidence_chunk"],
)

print("Docs-stage answer:", test_result["docs_stage_answer"])
print("Docs-stage facts preview:", test_result["docs_stage_facts"][:500])
print("Final LLM response:", test_result["response"])
print("Total input tokens across stages:", test_result["total_input_tokens"])
print("Was any stage truncated:", test_result["was_truncated"])

Test dataset: hotpotqa
Test question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
Test GT answer: Bedknobs and Broomsticks
Note: GT is printed only for human checking and is not sent to the LLM.
Test docs-stage input tokens: 2756
Test docs-stage was truncated: False
Test number of evidence chunks: 6
Docs-stage answer: Bedknobs and Broomsticks
Docs-stage facts preview: Bedknobs and Broomsticks was released in 1971 [Doc 1], while The Muppet Christmas Carol is a 1992 film [Doc 4]. Therefore, Bedknobs and Broomsticks is the older film.
Intermediate answer: Bedknobs and Broomsticks
Final LLM response: Bedknobs and Broomsticks
Total input tokens across stages: 3158
Was any stage truncated: False


In [11]:
#cell 10
# Process one dataset and save answers.

def get_processing_slice(records: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Return the selected processing slice."""
    end = ANSWER_END_INDEX if ANSWER_END_INDEX is not None else len(records)
    return records[ANSWER_START_INDEX:end]

def process_dataset(dataset_name: str, records: List[Dict[str, Any]], answer_path: Path) -> Dict[str, Any]:
    """
    Process one dataset independently.

    Important:
    - Only question and evidence_chunk are given to the LLM.
    - answer/supports/type are not included in the prompt.
    - evidence_chunk is capped at MAX_EVIDENCE_CHUNKS by validation/builders.
    - The final saved JSON contains exactly: type, question, gt, response.
    """
    print("=" * 80)
    print(f"Starting dataset: {dataset_name}")
    print("Answer path:", answer_path)
    print("Repository two-stage answer:", USE_REPOSITORY_TWO_STAGE)
    print("Model:", LLM_MODEL_NAME)
    print("Non-thinking mode:", True)

    selected_records = get_processing_slice(records)

    if not selected_records:
        raise ValueError(f"{dataset_name}: selected record slice is empty.")

    answer_records = []

    if RESUME_IF_EXISTS and answer_path.exists():
        existing = load_existing_answers(answer_path)

        # Resume only if the existing file matches the selected prefix.
        can_resume = True

        if len(existing) > len(selected_records):
            can_resume = False
        else:
            for i, old in enumerate(existing):
                if old.get("question") != selected_records[i].get("question"):
                    can_resume = False
                    break

        if can_resume:
            answer_records = existing
            print(f"{dataset_name}: resuming from {len(answer_records)} existing answers.")
        else:
            print(f"{dataset_name}: existing answer file does not match current data. Starting over.")

    start_time = time.time()

    input_token_counts = []
    truncation_count = 0
    num_docs_counts = []

    start_i = len(answer_records)

    for local_i in tqdm(
        range(start_i, len(selected_records)),
        desc=f"Answering {dataset_name}"
    ):
        rec = selected_records[local_i]

        llm_result = call_llm_answer(
            question=rec["question"],
            evidence_chunks=rec["evidence_chunk"],
        )

        input_token_counts.append(llm_result["total_input_tokens"])
        num_docs_counts.append(llm_result["num_docs"])

        if llm_result.get("was_truncated", False):
            truncation_count += 1

        output_record = {
            "type": rec["type"],
            "question": rec["question"],
            "gt": rec["answer"],
            "response": llm_result["response"],
        }

        validate_answer_record(output_record, dataset_name, local_i)
        answer_records.append(output_record)

        if len(answer_records) % SAVE_EVERY_N == 0:
            atomic_write_json(answer_records, answer_path)

        if CLEAR_CACHE_EVERY_N and len(answer_records) % CLEAR_CACHE_EVERY_N == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    atomic_write_json(answer_records, answer_path)

    elapsed = time.time() - start_time

    # Validate saved file.
    saved = load_existing_answers(answer_path)

    if len(saved) != len(selected_records):
        raise ValueError(
            f"{dataset_name}: saved {len(saved)} answers, "
            f"but expected {len(selected_records)}."
        )

    for i, out_rec in enumerate(saved):
        validate_answer_record(out_rec, dataset_name, i)

    summary = {
        "dataset": dataset_name,
        "num_input_records": len(records),
        "num_processed_records": len(saved),
        "answer_path": str(answer_path),
        "elapsed_seconds": elapsed,
        "num_truncated_prompts": truncation_count,
        "max_input_tokens_seen": max(input_token_counts) if input_token_counts else None,
        "mean_input_tokens_seen": float(sum(input_token_counts) / len(input_token_counts)) if input_token_counts else None,
        "min_num_docs_seen": min(num_docs_counts) if num_docs_counts else None,
        "max_num_docs_seen": max(num_docs_counts) if num_docs_counts else None,
    }

    print("=" * 80)
    print(f"{dataset_name}: done.")
    print("Saved to:", answer_path)
    print("Processed records:", len(saved))
    print("Elapsed seconds:", elapsed)
    print("Truncated prompts:", truncation_count)
    print("Docs seen min/max:", summary["min_num_docs_seen"], summary["max_num_docs_seen"])
    print("First output record:", saved[0])

    return summary

print("Dataset processing function is ready.")

Dataset processing function is ready.


In [12]:
#cell 11
# Run both datasets separately and save two independent JSON files.

run_summaries = []

for dataset_name in ["hotpotqa", "2wikimultihopqa"]:
    summary = process_dataset(
        dataset_name=dataset_name,
        records=evidence_data[dataset_name],
        answer_path=DATASETS[dataset_name]["answer_path"],
    )
    run_summaries.append(summary)

summary_df = pd.DataFrame(run_summaries)
display(summary_df)

print("All datasets are complete.")

Starting dataset: hotpotqa
Answer path: /content/drive/MyDrive/final_project/pyrag/answers/hotpotqa_gemma4_answers.json
Repository two-stage answer: True
Model: google/gemma-4-26B-A4B-it
Non-thinking mode: True


Answering hotpotqa:   0%|          | 0/1000 [00:00<?, ?it/s]

hotpotqa: done.
Saved to: /content/drive/MyDrive/final_project/pyrag/answers/hotpotqa_gemma4_answers.json
Processed records: 1000
Elapsed seconds: 1399.448426246643
Truncated prompts: 0
Docs seen min/max: 6 6
First output record: {'type': 'comparison', 'question': 'Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?', 'gt': 'Bedknobs and Broomsticks', 'response': 'Bedknobs and Broomsticks'}
Starting dataset: 2wikimultihopqa
Answer path: /content/drive/MyDrive/final_project/pyrag/answers/2wikimultihopqa_gemma4_answers.json
Repository two-stage answer: True
Model: google/gemma-4-26B-A4B-it
Non-thinking mode: True


Answering 2wikimultihopqa:   0%|          | 0/1000 [00:00<?, ?it/s]

2wikimultihopqa: done.
Saved to: /content/drive/MyDrive/final_project/pyrag/answers/2wikimultihopqa_gemma4_answers.json
Processed records: 1000
Elapsed seconds: 1533.95103764534
Truncated prompts: 0
Docs seen min/max: 6 6
First output record: {'type': 'bridge_comparison', 'question': 'Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?', 'gt': 'Kamakalawa', 'response': 'unknown'}


,dataset,num_input_records,num_processed_records,answer_path,elapsed_seconds,num_truncated_prompts,max_input_tokens_seen,mean_input_tokens_seen,min_num_docs_seen,max_num_docs_seen
0,hotpotqa,1000,1000,/content/drive/MyDrive/final_project/pyrag/ans...,1399.448426,0,3894,2942.102,6,6
1,2wikimultihopqa,1000,1000,/content/drive/MyDrive/final_project/pyrag/ans...,1533.951038,0,3810,2640.806,6,6


All datasets are complete.


In [13]:
#cell 12
# Verify final output files.

def verify_final_answer_file(dataset_name: str, answer_path: Path) -> None:
    """Verify final answer JSON format."""
    records = load_existing_answers(answer_path)

    if not records:
        raise ValueError(f"{dataset_name}: answer file is empty: {answer_path}")

    for i, rec in enumerate(records):
        validate_answer_record(rec, dataset_name, i)

    print("=" * 80)
    print(f"{dataset_name}: final answer file verified.")
    print("Path:", answer_path)
    print("Number of records:", len(records))
    print("First record keys:", list(records[0].keys()))
    print("First question:", records[0]["question"])
    print("First GT:", records[0]["gt"])
    print("First response:", records[0]["response"])

for dataset_name, cfg in DATASETS.items():
    verify_final_answer_file(dataset_name, cfg["answer_path"])

print("\nAnswer folder contents:")
for item in sorted(DRIVE_ANSWER_DIR.iterdir()):
    print(" -", item.name)

hotpotqa: final answer file verified.
Path: /content/drive/MyDrive/final_project/pyrag/answers/hotpotqa_gemma4_answers.json
Number of records: 1000
First record keys: ['type', 'question', 'gt', 'response']
First question: Which musical fantasy film is older, Bedknobs and Broomsticks or The Muppet Christmas Carol?
First GT: Bedknobs and Broomsticks
First response: Bedknobs and Broomsticks
2wikimultihopqa: final answer file verified.
Path: /content/drive/MyDrive/final_project/pyrag/answers/2wikimultihopqa_gemma4_answers.json
Number of records: 1000
First record keys: ['type', 'question', 'gt', 'response']
First question: Which film has the director died earlier, John Jaffer Janardhanan or Kamakalawa?
First GT: Kamakalawa
First response: unknown

Answer folder contents:
 - 2wikimultihopqa_gemma4_answers.json
 - 2wikimultihopqa_gpt_oss_120b_answers.json
 - 2wikimultihopqa_qwen3.5_answers.json
 - hotpotqa_gemma4_answers.json
 - hotpotqa_gpt_oss_120b_answers.json
 - hotpotqa_qwen3.5_answers.